In [ ]:
from weather_engine.database import engine
import xgboost as xgb
from pathlib import Path
import pandas as pd
from weather_engine.cell_interpolation import load_cell_features
from weather_engine.utils import get_project_root, encode_time_features

root = get_project_root()
models_dir = root / "models"
models_forecast_dir = models_dir / "single_point"
models_interpolate_dir = models_dir / "spatial_interpolation"

models_forecast = list(models_forecast_dir.glob("*.json"))
models_interpolate = list(models_interpolate_dir.glob("*.json"))


print(f"forecasting models: {models_forecast}\n interpolation models: {models_interpolate}")
print(models_interpolate[5].stem.split("xgb_")[1])

forecasting models: [PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/single_point/xgb_t+3.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/single_point/xgb_t+1.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/single_point/xgb_t+12.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/single_point/xgb_t+6.json')]
 interpolation models: [PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/spatial_interpolation/xgb_rh.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/spatial_interpolation/xgb_rain.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/spatial_interpolation/xgb_td.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/spatial_interpolation/xgb_ws.json'), PosixPath('/home/jadosh/repo/SparseData-AI-Precipitation-Forecasting/models/spat

In [12]:
cell_neighbors = pd.read_sql("SELECT * FROM cell_neighbors", engine)
all_data = pd.read_sql("SELECT * FROM clean_station_data", engine)
all_data['timestamp'] = pd.to_datetime(all_data['timestamp'])

all_data = all_data.set_index('timestamp').sort_index()
station_frames = {sid: grp.drop(columns='station_id') 
                  for sid, grp in all_data.groupby('station_id')}

In [16]:
models = {}
for model_path in models_interpolate:
    feature = model_path.stem.split("xgb_")[1]
    m = xgb.Booster()
    m.load_model(model_path)
    models[feature] = m

records = []

for _, row in cell_neighbors.iterrows():
    X = load_cell_features(
        row['elevation'],
        row['dist_to_coast'],
        int(row['neighbor_1_id']),
        int(row['neighbor_2_id']),
        int(row['neighbor_3_id']),
        row['neighbor_1_distance'],
        row['neighbor_2_distance'],
        row['neighbor_3_distance'],
        station_frames=station_frames
    )
    X = encode_time_features(X)

    record = {'cell_id': int(row['cell_id']), 'timestamp': X.index}
    for feature, model in models.items():
        record[feature] = model.predict(xgb.DMatrix(X[model.feature_names]))

    records.append(pd.DataFrame(record))

interpolated = pd.concat(records)
interpolated.to_sql("cell_interpolated", engine, if_exists="append", index=False)


3220839

In [22]:
from weather_engine.cell_forecasting import make_inference_features

models = {}
for model_path in models_forecast:
    _, step = model_path.stem.split("xgb_")[1].split("+")
    key = f"precipitation_t{step}"
    m = xgb.Booster()
    m.load_model(model_path)
    print(m.feature_names)
    models[key] = m

upstream_dfs = {
    "tel_aviv": station_frames[178],
    "haifa": station_frames[43],
}

records = []

for _, row in cell_neighbors.iterrows():
    cell_id = int(row['cell_id'])
    df_target = pd.read_sql(
        "SELECT * FROM cell_interpolated WHERE cell_id = :cid",
        engine, params={'cid': cell_id},
    )
    df_target['timestamp'] = pd.to_datetime(df_target['timestamp'])
    df_target = df_target.drop(columns=['cell_id']).set_index('timestamp').sort_index()

    X = make_inference_features(df_target, upstream_dfs)

    record = {'cell_id': cell_id, 'timestamp': X.index}
    for key, model in models.items():
        record[key] = model.predict(xgb.DMatrix(X[model.feature_names]))

    records.append(pd.DataFrame(record))

forecasts = pd.concat(records)
forecasts.to_sql("cell_forecasts", engine, if_exists="append", index=False)



['rain', 'ws', 'td', 'rh', 'tdmax', 'tdmin', 'u_vec', 'v_vec', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'rain_t-1h', 'rain_t-2h', 'rain_t-3h', 'rain_t-6h', 'rain_t-12h', 'rain_t-24h', 'u_vec_t-1h', 'u_vec_t-2h', 'u_vec_t-3h', 'u_vec_t-6h', 'u_vec_t-12h', 'u_vec_t-24h', 'v_vec_t-1h', 'v_vec_t-2h', 'v_vec_t-3h', 'v_vec_t-6h', 'v_vec_t-12h', 'v_vec_t-24h', 'td_t-1h', 'td_t-2h', 'td_t-3h', 'td_t-6h', 'td_t-12h', 'td_t-24h', 'rh_t-1h', 'rh_t-2h', 'rh_t-3h', 'rh_t-6h', 'rh_t-12h', 'rh_t-24h', 'rain_tel_aviv', 'u_vec_tel_aviv', 'v_vec_tel_aviv', 'rh_tel_aviv', 'u_convergence_tel_aviv', 'v_convergence_tel_aviv', 'moisture_flux_tel_aviv', 'rain_tel_aviv_t-1h', 'rain_tel_aviv_t-2h', 'rain_tel_aviv_t-3h', 'u_vec_tel_aviv_t-1h', 'u_vec_tel_aviv_t-2h', 'u_vec_tel_aviv_t-3h', 'v_vec_tel_aviv_t-1h', 'v_vec_tel_aviv_t-2h', 'v_vec_tel_aviv_t-3h', 'rh_tel_aviv_t-1h', 'rh_tel_aviv_t-2h', 'rh_tel_aviv_t-3h', 'u_convergence_tel_aviv_t-1h', 'u_convergence_tel_aviv_t-2h', 'u_convergence_tel_aviv_t-3h'

3219327